In [1]:
from html import escape
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from plotnine import ggplot, aes, geom_map, geom_point, scale_fill_manual, coord_fixed, theme_void
from ninejs import interactive, css, save, show

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
# Rossio Metro coords
lat = 38.713905
lon = -9.139164
crs = "EPSG:3763"

In [3]:
# Load data
streets = gpd.read_file("rossio_streets.geojson")
pois = gpd.read_file("rossio_pois.geojson")
isochrone_gdf = gpd.read_file("rossio_isochrone.geojson")
reachable_gdf = gpd.read_file("rossio_reachable_streets.geojson")

In [4]:
# Project to metric CRS
streets = streets.to_crs(crs)
pois = pois.to_crs(crs)
isochrone_gdf = isochrone_gdf.to_crs(crs)
reachable_gdf = reachable_gdf.to_crs(crs)

In [5]:
# Starting point
start = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326").to_crs(crs).iloc[0]

In [6]:
# POIs
pois = pois[pois["name"].notna()].copy()
pois["geometry"] = pois.geometry.representative_point()

# Accessibility polygon
isochrone = isochrone_gdf.geometry.union_all()

# POIs inside the polygon
pois = pois[pois.geometry.intersects(isochrone)].copy()

# Remove duplicate names
pois = pois.drop_duplicates(subset="name")

# Museums only
pois = pois[pois["tourism"] == "museum"].reset_index(drop=True).copy()

# Tooltip df
df = pd.read_csv("museums3.csv")

# Merge POIs with tooltip
pois = pois.merge(df, on="name", how="left")

In [7]:
# Museum data
museum_data = pd.DataFrame({
    "x": pois.geometry.x,
    "y": pois.geometry.y,
    "name": pois["name"],
    "tooltip": pois["tooltip"]
})

museum_data["tooltip"] = museum_data["tooltip"].fillna("")

# Tooltip 
museum_data["tooltip"] = museum_data.apply(
    lambda row: (
        f"<strong>{escape(str(row["name"]))}</strong>"
        f"<div>{escape(str(row["tooltip"]))}</div>"
    ),
    axis=1
)

museum_data["type"] = "museum"

In [8]:
# Rossio metro
rossio_data = pd.DataFrame({
    "x": [start.x],
    "y": [start.y],
    "name": ["Rossio Metro"],
    "tooltip": [
        "<strong>Rossio Metro</strong>"
        "<div>Starting point</div>"
    ],
    "type": ["station"]
})

# Add Rossio to tooltip
point_data = pd.concat([museum_data, rossio_data], ignore_index=True)

In [9]:
# Tooltip css
tooltip_css = """
.tooltip {
    max-width: 300px;
    white-space: normal;
    overflow-wrap: break-word;
    line-height: 1.5;
    font-size: 13px;
    padding: 10px 12px;
    background: white;
    color: #222;
    border: 1px solid #555;
    border-radius: 6px;
    box-shadow: 0 2px 8px rgba(0,0,0,0.15);
}
/* Prevent point markers from fading during hover */
.point {
    opacity: 1 !important;
}
.tooltip strong {
    display: block;
    margin-bottom: 6px;
    font-size: 14px;
}
/* Control the interactive map size */
#plot-container {
    width: 1000px;
    max-width: 95vw;
    margin: 0 auto;
}

#plot-container svg {
    width: 100%;
    height: auto;
    display: block;
}
"""

In [10]:
xmin, ymin, xmax, ymax = isochrone.bounds
margin = 100 

gg = (
    ggplot()
    # Streets
    + geom_map(data=streets, color="#272a24", size=0.3)
    # Polygon
    + geom_map(data=isochrone_gdf, fill="#33AFDB", color="#33AFDB", alpha=0.7, size=0.6)
    # Streets inside
    + geom_map(data=reachable_gdf, color="#004E78", size=0.4)
    # Interactive points
    + geom_point(
        data=point_data,
        mapping=aes(x="x", y="y", tooltip="tooltip", fill="type"),
        color="#272a24", shape="o", size=4, stroke=0.05)
    # Marker colors
    + scale_fill_manual(
        values={
            "museum": "#FFD43B",
            "station": "#9E1C0B"
        },
        guide=None
    )
    # Map coords
    + coord_fixed(xlim=(xmin - margin, xmax + margin), ylim=(ymin - margin, ymax + margin), expand=False)
    + theme_void()    
)

fig = gg.draw()
fig.set_facecolor("#dde0da")
fig.set_size_inches(7, 7, forward=True)
ax = fig.axes[0]
ax.set_aspect("equal")
ax.axis("off")

# Title
fig.text(0.5, 0.95,
         "Lisbon in Sight",
         color="#272a24", size=25, weight="bold", ha="center")

# Subtitle
fig.text(0.5, 0.9,
         "Museums accessible within a 10-minute walk",
         color="#272a24", size=10, ha="center")

# Caption
fig.text(0.5, 0.1,
         "Data: OpenStreetMap (licensed under ODbL) | Map: Bernadett Piros",
         color="#272a24", size=6, ha="center")

# Png
fig.savefig("rossio_museums.png", bbox_inches="tight", pad_inches=0.2)

# Html
chart = interactive(gg) + css(tooltip_css)
chart + save("rossio_museums.html")